# Clasificación de Noticias

## Data Preparation

In [4]:
from gensim.models import KeyedVectors
import os

w2v_model = KeyedVectors.load('../Representacion_del_lenguaje/embeddings/NoContext/w2v_sg.kv', mmap='r')
print(w2v_model)

KeyedVectors<vector_size=300, 16863 keys>


In [5]:
import pandas as pd

processed_texts = pd.read_parquet('../Representacion_del_lenguaje/processed_data/preprocNoContext_embeddings.parquet')
processed_texts = processed_texts['ncEmbedText']
processed_texts.head()

0    the jobs market continues show signs weakness ...
1    new data from the department from work and pen...
2    asking for workplace accommodations often easi...
3    apple has announced major expansion its renewa...
4    the advent artificial intelligence less than t...
Name: ncEmbedText, dtype: object

In [6]:
import numpy as np
import nltk

MAX_LEN = 250   # Tú decides cuántas palabras mirar por texto
EMBEDDING_DIM = 300 # Tamaño de tu Word2Vec/FastText

def crear_secuencias_vectores(texts, modelo_w2v, max_len):
    textos_tokenizados = [nltk.word_tokenize(texto.lower()) for texto in texts]
    num_textos = len(textos_tokenizados)
    # Creamos una matriz gigante de ceros
    # (Numero Textos, Longitud Maxima, 300)
    data_matrix = np.zeros((num_textos, max_len, EMBEDDING_DIM))
    
    for i, texto in enumerate(textos_tokenizados):
        # Para cada palabra en el texto
        for j, palabra in enumerate(texto):
            if j >= max_len:
                break # Si el texto es muy largo, paramos
            
            # Si la palabra existe en el modelo, ponemos su vector
            if palabra in modelo_w2v:
                data_matrix[i, j, :] = modelo_w2v[palabra]
            # Si no existe, se queda en ceros (padding implícito)
            
    return data_matrix

In [7]:
X_w2v = crear_secuencias_vectores(processed_texts, w2v_model, MAX_LEN)

print(X_w2v.shape)

(5160, 250, 300)


In [8]:
print(X_w2v[0])

[[-0.10494587  0.00571608 -0.02699127 ... -0.06973305 -0.11877021
  -0.06472845]
 [-0.16664293  0.3422474   0.37715384 ... -0.19503252  0.08127215
  -0.37072948]
 [ 0.21194062  0.02457219 -0.34651375 ...  0.03259023 -0.29865226
  -0.21168469]
 ...
 [-0.09888955 -0.0249728   0.14641783 ...  0.01026709  0.01079688
   0.287429  ]
 [ 0.25404429  0.23451392  0.52566278 ...  0.24864739 -0.0508902
   0.38398963]
 [ 0.20846635  0.04844557  0.3349756  ... -0.13310026 -0.08682542
   0.1327893 ]]


In [9]:
topics = pd.read_csv("../../data/definitivos/INDEX_ALL_scrapped_filtrado.csv")
y = topics["topic"].values

print("Total de textos:", y.size)
print("\nDistribución de textos por topic:")
print(topics["topic"].value_counts().sort_index())

Total de textos: 5160

Distribución de textos por topic:
topic
Business Growth and Cloud Infrastructure in the AI Industry    1539
Financial and Market News and Corporate Sales                  2257
Informal / Conversational Lenguaje                              152
Quantum Computing and Military Technology                       101
Stock Market and Trading                                       1111
Name: count, dtype: int64


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Codificar las etiquetas de texto a números (0, 1, 2, 3, 4)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Clases codificadas:")
for i, clase in enumerate(label_encoder.classes_):
    print(f"{i}: {clase}")

xw2v_train_global, xw2v_test_global, y_train_global, y_test_global = train_test_split(X_w2v, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("\nTextos (variable predictoria):\n", "Training", xw2v_train_global.shape, "Test", xw2v_test_global.shape)
print("Topics (variable a predecir):\n", "Training", y_train_global.shape, "Test", y_test_global.shape)

Clases codificadas:
0: Business Growth and Cloud Infrastructure in the AI Industry
1: Financial and Market News and Corporate Sales
2: Informal / Conversational Lenguaje
3: Quantum Computing and Military Technology
4: Stock Market and Trading

Textos (variable predictoria):
 Training (4128, 250, 300) Test (1032, 250, 300)
Topics (variable a predecir):
 Training (4128,) Test (1032,)


## Convolutional Neural Network

In [13]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Conv1D, GlobalMaxPooling1D
from keras.optimizers import Adam

def modelo_cnn(input_shape, num_filtros=128, kernel_size=3, lr=0.001, dropout=0.5):
    model = Sequential()

    # --- BLOQUE CONVOLUCIONAL ---
    # Conv1D: Mira grupos de palabras (kernel_size=3 es como trigramas)
    model.add(Conv1D(filters=num_filtros, 
                     kernel_size=kernel_size, 
                     activation='relu', 
                     input_shape=input_shape))
    
    # Reduce la dimensionalidad quedándose con el valor más alto (el rasgo más fuerte)
    model.add(GlobalMaxPooling1D())

    # --- BLOQUE DENSO (Clasificación) ---
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(dropout)) # Apaga neuronas al azar para evitar memorizar

    # Salida
    model.add(Dense(5, activation='softmax'))

    # Compilación
    optimizer = Adam(learning_rate=lr)
    model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    return model

In [16]:
from sklearn.model_selection import KFold
import numpy as np

# Configuración del CV
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
resultados_accuracy = []

# Dimensiones de entrada para la CNN (Ej: 50 pasos, 300 features)
input_shape = (xw2v_train_global.shape[1], xw2v_train_global.shape[2])

print(f"Iniciando CV con input shape: {input_shape}...")

fold_idx = 1
for train_index, val_index in kf.split(xw2v_train_global):
    print(f"\n--- Fold {fold_idx}/{n_folds} ---")
    
    # 1. Split de datos para este Fold
    X_train_f, X_val_f = xw2v_train_global[train_index], xw2v_train_global[val_index]
    y_train_f, y_val_f = y_train_global[train_index], y_train_global[val_index]
    
    # 2. Crear modelo nuevo
    model = modelo_cnn(input_shape, num_filtros=128, kernel_size=5, dropout=0.5)
    
    # 3. Entrenar
    # Tip: epochs bajas en CV para no tardar una eternidad probando
    model.fit(X_train_f, y_train_f, 
              validation_data=(X_val_f, y_val_f),
              epochs=5, 
              batch_size=32, 
              verbose=1) # Pon 0 si quieres silenciar la salida
    
    # 4. Evaluar
    loss, accuracy = model.evaluate(X_val_f, y_val_f, verbose=0)
    resultados_accuracy.append(accuracy)
    print(f"Accuracy del Fold: {accuracy:.4f}")
    
    fold_idx += 1

print("\n" + "="*30)
print(f"ACCURACY PROMEDIO: {np.mean(resultados_accuracy):.4f} (+/- {np.std(resultados_accuracy):.4f})")

Iniciando CV con input shape: (250, 300)...

--- Fold 1/5 ---


c:\Users\Iñigo Peña\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.5636 - loss: 1.0784 - val_accuracy: 0.7446 - val_loss: 0.7419
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.5636 - loss: 1.0784 - val_accuracy: 0.7446 - val_loss: 0.7419
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7707 - loss: 0.6496 - val_accuracy: 0.8148 - val_loss: 0.5195
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7707 - loss: 0.6496 - val_accuracy: 0.8148 - val_loss: 0.5195
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8428 - loss: 0.4516 - val_accuracy: 0.8426 - val_loss: 0.4483
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8428 - loss: 0.4516 - val_accuracy: 0.8426 - val_loss: 0.4483
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9110 - loss: 0.2802 - val_accuracy: 0.8462 - val_loss: 0.4154
Epoch 5/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9110 - loss: 0.2802 - val_accuracy: 0.